# Week 2: Pandas 强化训练 (Muscle Memory Drills)

> **💡 训练目标**：
> 彻底粉碎 Week 1 的“循环思维”。通过高强度的重复练习，形成 **“列思维” (Vectorization)** 的肌肉记忆。
> 
> **核心规则**：
> ❌ **本文件内严禁使用 `for`, `while`, `if` 语句！**
> ✅ 必须使用 Pandas 的内置整列操作。

In [7]:
import pandas as pd
import numpy as np

# 🛒 准备Mock数据: 某电商平台的订单表
df = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Laptop', 'Screen', 'Mouse', 'Keyboard', 'Screen'],
    'category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics', 'Accessories', 'Accessories', 'Electronics'],
    'price': [5000, 100, 300, 5200, 1500, 120, 280, 1600],
    'quantity': [1, 5, 2, 1, 2, 10, 1, 3],
    'discount_rate': [0.1, 0.0, 0.2, 0.1, 0.05, 0.1, 0.0, 0.1],  # 0.1 代表 10% 折扣
    'city': ['Beijing', 'Shanghai', 'Beijing', 'Shenzhen', 'Shanghai', 'Beijing', 'Shenzhen', 'Shanghai']
})

print("强化训练数据已加载！")
print(df.head())
print("===info===")
print(df.info())
print("===describe===")
print(df.describe())
print("===shape===")
print(df.shape)

强化训练数据已加载！
   order_id   product     category  price  quantity  discount_rate      city
0       101    Laptop  Electronics   5000         1           0.10   Beijing
1       102     Mouse  Accessories    100         5           0.00  Shanghai
2       103  Keyboard  Accessories    300         2           0.20   Beijing
3       104    Laptop  Electronics   5200         1           0.10  Shenzhen
4       105    Screen  Electronics   1500         2           0.05  Shanghai
===info===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       8 non-null      int64  
 1   product        8 non-null      object 
 2   category       8 non-null      object 
 3   price          8 non-null      int64  
 4   quantity       8 non-null      int64  
 5   discount_rate  8 non-null      float64
 6   city           8 non-null      object 
dtypes: float64(1), 

## 🏋️‍♂️ 训练 1: 向量化计算 (Broadcasting)
**场景**：计算每一单的 **实际成交金额**。
*   公式：`GMV = price * quantity * (1 - discount_rate)`
*   **挑战**：一行代码搞定，不许用循环。

In [12]:
# ✏️ 在这里写你的代码
# df['real_amount'] = ...
df['gmv']=df['price']*df['quantity']*(1-df['discount_rate'])
# gmv
df

,order_id,product,category,price,quantity,discount_rate,city,gmv
0,101,Laptop,Electronics,5000,1,0.10,Beijing,4500.0
1,102,Mouse,Accessories,100,5,0.00,Shanghai,500.0
2,103,Keyboard,Accessories,300,2,0.20,Beijing,480.0
3,104,Laptop,Electronics,5200,1,0.10,Shenzhen,4680.0
4,105,Screen,Electronics,1500,2,0.05,Shanghai,2850.0
5,106,Mouse,Accessories,120,10,0.10,Beijing,1080.0
6,107,Keyboard,Accessories,280,1,0.00,Shenzhen,280.0
7,108,Screen,Electronics,1600,3,0.10,Shanghai,4320.0


In [ ]:
# 🔑 参考答案
df['real_amount'] = df['price'] * df['quantity'] * (1 - df['discount_rate'])
df

## 🏋️‍♂️ 训练 2: 复杂逻辑筛选 (Complex Filtering)
**场景**：找出需要优先发货的“高价值订单”。
**条件**（必须同时满足）：
1.  城市是 'Beijing' **或者** 'Shanghai'
2.  **并且** 购买数量 (`quantity`) 大于 2

*   **提示**：注意 `&` (And) 和 `|` (Or) 的优先级，**括号**不能省！
*   **高级提示**：城市筛选可以用 `.isin(['Beijing', 'Shanghai'])`

In [17]:
# ✏️ 在这里写你的代码
df['high_value'] = ((df['city'] == 'Beijing')| (df['city'] == 'Shanghai')) & (df['quantity']>2)
# print(high_value)
df

,order_id,product,category,price,quantity,discount_rate,city,gmv,high_value
0,101,Laptop,Electronics,5000,1,0.10,Beijing,4500.0,False
1,102,Mouse,Accessories,100,5,0.00,Shanghai,500.0,True
2,103,Keyboard,Accessories,300,2,0.20,Beijing,480.0,False
3,104,Laptop,Electronics,5200,1,0.10,Shenzhen,4680.0,False
4,105,Screen,Electronics,1500,2,0.05,Shanghai,2850.0,False
5,106,Mouse,Accessories,120,10,0.10,Beijing,1080.0,True
6,107,Keyboard,Accessories,280,1,0.00,Shenzhen,280.0,False
7,108,Screen,Electronics,1600,3,0.10,Shanghai,4320.0,True


In [ ]:
# 🔑 参考答案
# 方法 1: 笨办法
# df[ ((df['city'] == 'Beijing') | (df['city'] == 'Shanghai')) & (df['quantity'] > 2) ]

# 方法 2: 优雅写法 (isin)
busy_orders = df[ df['city'].isin(['Beijing', 'Shanghai']) & (df['quantity'] > 2) ]
busy_orders

## 🏋️‍♂️ 训练 3: 多维分组统计 (Multi-Level GroupBy)
**场景**：按 **城市** 和 **商品类别** 两个维度拆解销售情况。
**任务**：
1.  分组：`['city', 'category']`
2.  计算：`quantity` 的总和 (`sum`) 和平均值 (`mean`)

*   **SQL脑补**：`GROUP BY city, category`

In [21]:
# ✏️ 在这里写你的代码
df.groupby(['city','category'],as_index=False)['quantity'].agg(['sum','mean'])


,city,category,sum,mean
0,Beijing,Accessories,12,6.0
1,Beijing,Electronics,1,1.0
2,Shanghai,Accessories,5,5.0
3,Shanghai,Electronics,5,2.5
4,Shenzhen,Accessories,1,1.0
5,Shenzhen,Electronics,1,1.0


In [ ]:
# 🔑 参考答案
report = df.groupby(['city', 'category'])['quantity'].agg(['sum', 'mean'])
report

## 🏋️‍♂️ 训练 4: 排序与Top N (Sorting)
**场景**：谁是“销量之王”？
**任务**：
1.  将数据按 `price` (单价) **从高到低** 排序。
2.  取前 3 名。

In [24]:
# ✏️ 在这里写你的代码
df['rank']=df['price'].rank(ascending=False,method='min')

df

,order_id,product,category,price,quantity,discount_rate,city,gmv,high_value,rank
0,101,Laptop,Electronics,5000,1,0.10,Beijing,4500.0,False,2.0
1,102,Mouse,Accessories,100,5,0.00,Shanghai,500.0,True,8.0
2,103,Keyboard,Accessories,300,2,0.20,Beijing,480.0,False,5.0
3,104,Laptop,Electronics,5200,1,0.10,Shenzhen,4680.0,False,1.0
4,105,Screen,Electronics,1500,2,0.05,Shanghai,2850.0,False,4.0
5,106,Mouse,Accessories,120,10,0.10,Beijing,1080.0,True,7.0
6,107,Keyboard,Accessories,280,1,0.00,Shenzhen,280.0,False,6.0
7,108,Screen,Electronics,1600,3,0.10,Shanghai,4320.0,True,3.0


## 🔥 训练 5: 窗口函数 (Window Functions)

**这是您之前疑惑的点！**

**场景**：你想算每个订单在**它所属城市**的总销量里占多少比例？
*   **SQL 思维**：`quantity / SUM(quantity) OVER (PARTITION BY city)`
*   **Pandas 神器**：`transform`

**逻辑**：
1.  先 `groupby('city')`。
2.  然后用 `transform('sum')` —— 这句话的意思是：“去把这一组的和算出来，**然后把这个总数贴回每一行后面**”。
3.  最后做除法。

In [28]:
# 1. 算出每个城市总销量，并贴回每一行 (注意长度不变！)
df['city_total'] = df.groupby('city')['quantity'].transform('sum')
# df
# 看看它长啥样
# print(city_total)

# 2. 算占比
df['city_share'] = df['quantity'] / df['city_total']
df

,order_id,product,category,price,quantity,discount_rate,city,gmv,high_value,rank,city_share,city_total
0,101,Laptop,Electronics,5000,1,0.10,Beijing,4500.0,False,2.0,0.076923,13
1,102,Mouse,Accessories,100,5,0.00,Shanghai,500.0,True,8.0,0.500000,10
2,103,Keyboard,Accessories,300,2,0.20,Beijing,480.0,False,5.0,0.153846,13
3,104,Laptop,Electronics,5200,1,0.10,Shenzhen,4680.0,False,1.0,0.500000,2
4,105,Screen,Electronics,1500,2,0.05,Shanghai,2850.0,False,4.0,0.200000,10
5,106,Mouse,Accessories,120,10,0.10,Beijing,1080.0,True,7.0,0.769231,13
6,107,Keyboard,Accessories,280,1,0.00,Shenzhen,280.0,False,6.0,0.500000,2
7,108,Screen,Electronics,1600,3,0.10,Shanghai,4320.0,True,3.0,0.300000,10
